# LLM Drives the Arm to a Towel

The canonical AI-engineer workflow: feed live ARM + perception data to an LLM, let it choose firmware commands, send them on `/arm/command`, and watch `/arm/state` + `/arm/response` until the towel is grasped.

Works fully offline with the deterministic `MockLLMClient`. Set `LLM_BASE_URL`, `LLM_API_KEY` (and optionally `LLM_MODEL`) to use any OpenAI-compatible endpoint instead.

Prerequisites: stack running (`docker compose up`), robot positioned so a towel is in the pick window — e.g. drive it manually in the FE, or let this notebook position it. `pip install websocket-client`.

In [ ]:
import json, sys, threading, time
import websocket
sys.path.insert(0, '..')  # onsen_ai_worker package
from onsen_ai_worker.llm_client import create_llm_client, complete_json

latest, responses = {}, []

def on_message(_ws, raw):
    msg = json.loads(raw)
    if msg.get('op') != 'publish':
        return
    if msg['topic'] == '/arm/response':
        responses.append(msg['msg']['data'])
    else:
        latest[msg['topic']] = msg['msg']

ws = websocket.WebSocketApp('ws://localhost:9090', on_message=on_message)
threading.Thread(target=ws.run_forever, daemon=True).start()
time.sleep(1)
for topic, typ in [('/arm/state', 'std_msgs/String'),
                   ('/arm/response', 'std_msgs/String'),
                   ('/detected_objects', 'std_msgs/String'),
                   ('/ground_truth/objects', 'std_msgs/String')]:
    ws.send(json.dumps({'op': 'subscribe', 'topic': topic, 'type': typ}))
ws.send(json.dumps({'op': 'advertise', 'topic': '/arm/command', 'type': 'std_msgs/String'}))

def arm(cmd):
    ws.send(json.dumps({'op': 'publish', 'topic': '/arm/command', 'msg': {'data': cmd}}))
    print('>', cmd)

def arm_status():
    return json.loads(latest.get('/arm/state', {}).get('data', '{}')).get('status', '?')

def wait_idle(timeout=8.0):
    t0 = time.time()
    time.sleep(0.3)
    while arm_status() != 'IDLE' and time.time() - t0 < timeout:
        time.sleep(0.2)
    return arm_status()

time.sleep(2)
print('arm status:', arm_status())

## 1. Ask the LLM what to do
The LLM sees the current detections (and arm state) and answers with a structured action.

In [ ]:
llm = create_llm_client()
print('LLM backend:', llm.name)

SYSTEM = (
    'You control a 6-axis arm on a towel-collecting robot. '
    'Reply with JSON {"action": "pick_object"|"continue_search", '
    '"target_id": str|null, "reason": str}.'
)
detections = json.loads(latest.get('/detected_objects', {}).get('data', '{}')).get('objects', [])
verdict = complete_json(llm, SYSTEM, json.dumps({'detections': detections, 'safety_stop': False}))
print(json.dumps(verdict, indent=2))

## 2. Execute the pick sequence the LLM decided on
The pose table lives in the arm firmware (`arm_protocol.py`); each step waits for the firmware to report IDLE — exactly how the mission executor does it.

In [ ]:
if verdict.get('action') == 'pick_object':
    for cmd in ['A PRE_PICK', 'A PICK_LOWER', 'A PICK_SCOOP',
                'A PICK_GRIP', 'A PICK_LIFT', 'A PICK_RETRACT']:
        arm(cmd)
        print('  status ->', wait_idle())
else:
    print('LLM chose not to pick:', verdict.get('reason'))
print('firmware replies:', responses[-8:])

## 3. Verify the grasp against simulation ground truth

In [ ]:
objects = json.loads(latest.get('/ground_truth/objects', {}).get('data', '{}')).get('objects', [])
held = [o for o in objects if o.get('held')]
print('held objects:', held or 'none — adjust robot position and retry')
arm('Q')
time.sleep(0.5)
print(responses[-1] if responses else 'no reply')

## 4. Raw command exploration
Everything in the firmware grammar works from here: `J 90 70 120 110 90 40 1000`, `D 0 -15 300`, `M BASE_LEFT 10 250`, `G 40 200`, `SPEED 50`, `STOP` / `A RESET_ERROR`.

In [ ]:
arm('J 90 70 120 110 90 40 1000')
print('  status ->', wait_idle())
arm('A HOME')
print('  status ->', wait_idle())
print(responses[-4:])